# Giorno 1 — Object Detection & Multi-Object Tracking

**Obiettivi della giornata:**
1. Rilevare oggetti in video usando YOLOv8 (modello pre-addestrato)
2. Tracciare più oggetti simultaneamente con BoxMOT
3. Simulare condizioni di acquisizione reali tramite degradazione controllata (OpenCV)
4. Confrontare i risultati dei punti 1-2 prima e dopo la degradazione

---
**Dataset:**
- `MOT20` — sequenze di persone in ambienti affollati
- `UA-DETRAC` (`MVI_*`) — sequenze di traffico veicolare

**Modelli:** YOLOv8n (nano, ~6MB) + BoxMOT (ByteTrack)

> In questo laboratorio utilizziamo esclusivamente modelli **pre-addestrati**: nessun training, solo inference.

## 0. Setup — Installazione e import

Eseguire questa cella **una sola volta** all'inizio della sessione.

In [ ]:
# Installazione delle dipendenze (e clone del repo su Google Colab)
import os, sys

if 'google.colab' in sys.modules:
    if not os.path.exists('/content/Corso-Computer-Vision'):
        !git clone https://github.com/SalvoSamba01/Corso-Computer-Vision
    %cd /content/Corso-Computer-Vision

!pip install -q ultralytics boxmot opencv-python-headless matplotlib tqdm

**Import librerie e utility.** Carica OpenCV, NumPy, Matplotlib e le utility personalizzate del corso (`cv_utils.py`). Vengono anche definite le funzioni `converti_video_h264()` e `mostra_video_colab()` per la riproduzione video inline nel notebook.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import subprocess
from pathlib import Path
from tqdm import tqdm
from IPython.display import HTML, display
import warnings
warnings.filterwarnings('ignore')

import sys, os
if os.path.exists('/content/Corso-Computer-Vision'):
    sys.path.append('/content/Corso-Computer-Vision/utils')
else:
    sys.path.append('../utils')
from cv_utils import (
    mostra_frame, mostra_confronto, mostra_griglia,
    stampa_info_video, estrai_frame,
    degrada_immagine, riduci_risoluzione, applica_blur, aggiungi_rumore,
    disegna_detection, disegna_tracks,
)


def converti_video_h264(input_path: str, output_path: str) -> str:
    """Re-encode video to H.264 for browser playback in Colab."""
    r = subprocess.run(
        ['ffmpeg', '-y', '-i', input_path, '-vcodec', 'libx264',
         '-crf', '23', '-movflags', '+faststart', output_path, '-loglevel', 'quiet'],
        capture_output=True
    )
    if r.returncode == 0 and os.path.exists(output_path) and os.path.getsize(output_path) > 0:
        if input_path != output_path:
            try:
                os.remove(input_path)
            except Exception:
                pass
        return output_path
    return input_path


def mostra_video_colab(path, width=700):
    """Embed a video inline in a Colab/Jupyter cell."""
    from base64 import b64encode
    with open(path, 'rb') as f:
        video_b64 = b64encode(f.read()).decode()
    display(HTML(
        f'<video width="{width}" controls>'
        f'<source src="data:video/mp4;base64,{video_b64}" type="video/mp4">'
        f'</video>'
    ))


print('✅ Import completati')

In [ ]:
# ── Configurazione percorsi ──────────────────────────────────────────────────
import os
from pathlib import Path

if os.path.exists('/content/Corso-Computer-Vision'):
    DATA_DIR = Path('/content/Corso-Computer-Vision/data/day_1')
    BASE_DIR  = Path('/content/Corso-Computer-Vision')
else:
    DATA_DIR = Path('../data/day_1')
    BASE_DIR  = Path('..')

MOT20_DIR  = DATA_DIR / 'MOT20'
DETRAC_DIR = DATA_DIR / 'UA-DETRAC'

VIDEOS_DIR = BASE_DIR / 'results' / 'videos'
VIDEOS_DIR.mkdir(parents=True, exist_ok=True)

# Verifica dei file disponibili
video_mot20  = sorted(MOT20_DIR.glob('*.mp4'))
video_detrac = sorted(DETRAC_DIR.glob('*.mp4'))
video_files  = video_mot20 + video_detrac

print(f'Video MOT20 trovati: {len(video_mot20)}')
for v in video_mot20:
    print(f'  {v.name}')
print(f'\nVideo UA-DETRAC trovati: {len(video_detrac)}')
for v in video_detrac:
    print(f'  {v.name}')
print(f'\nOutput video → {VIDEOS_DIR}')

---
## 1. Esplorazione del dataset

Prima di applicare qualsiasi modello, esploriamo i video: risoluzione, FPS, contenuto.

**I dataset che utilizziamo:**
- **MOT20** (`MOT20-*.mp4`): benchmark per il tracking di pedoni in scene affollate (stazioni, eventi pubblici)
- **UA-DETRAC** (`MVI_*.mp4`): traffico veicolare ripreso da telecamere di sorveglianza stradali

In [ ]:
# Stampa metadati di ogni video: risoluzione, FPS, durata.
# stampa_info_video() usa cv2.VideoCapture per leggere le proprietà senza decodificare i frame.
# Stampa le informazioni di tutti i video
print('=' * 55)
for v in video_files:
    print(f'\n📹 {v.name}')
    stampa_info_video(str(v))
print('=' * 55)


**Anteprima dataset.** Estrae il primo frame di ogni sequenza video per avere una visione d'insieme del contenuto prima di applicare qualsiasi elaborazione.

In [ ]:
# estrai_frame(path, n=1) usa cv2.VideoCapture e restituisce i primi n frame come array BGR.
# Visualizziamo il primo frame di ciascuna sequenza
frames_preview = []
titoli_preview = []

for v in video_files:
    frame = estrai_frame(str(v), n=1)[0]
    frames_preview.append(frame)
    titoli_preview.append(v.stem)

mostra_griglia(frames_preview, titoli_preview, colonne=2)

In [ ]:
# Selezioniamo un video per tipo come riferimento per gli esperimenti
VIDEO_PEDONI  = str(MOT20_DIR  / 'MOT20-01.mp4')   # benchmark pedoni (affollamento moderato)
VIDEO_VEICOLI = str(DETRAC_DIR / 'MVI_20035.mp4')  # traffico veicolare
VIDEO_FOLLA   = str(MOT20_DIR  / 'MOT20-06.mp4')   # folla densa — sequenza alternativa MOT20

print('Video selezionati per gli esperimenti:')
print(f'  Pedoni  → {Path(VIDEO_PEDONI).name}')
print(f'  Veicoli → {Path(VIDEO_VEICOLI).name}')
print(f'  Folla   → {Path(VIDEO_FOLLA).name}')

---
## 2. Object Detection con YOLOv8

**YOLO** (You Only Look Once) è una famiglia di modelli di object detection in tempo reale.
La versione 8 di Ultralytics è lo stato dell'arte per applicazioni pratiche: veloce, accurata, facile da usare.

**Come funziona (in breve):**
- Il modello divide l'immagine in una griglia
- Per ogni cella predice bounding box + classe + confidenza
- Una fase di NMS (Non-Maximum Suppression) elimina le detection ridondanti

Usiamo `YOLOv8n` (nano): il più leggero della famiglia, ideale per dimostrazione.
I pesi (~6MB) vengono scaricati automaticamente al primo utilizzo.

In [ ]:
from ultralytics import YOLO

# Caricamento del modello (download automatico al primo utilizzo)
# yolov8n = nano (~6MB), yolov8s = small (~22MB), yolov8m = medium (~52MB)
model_yolo = YOLO('yolov8n.pt')
print(f'Modello caricato: {model_yolo.model.__class__.__name__}')
print(f'Classi COCO: {len(model_yolo.names)} classi')

In [ ]:
# Stampa tutte le classi COCO riconosciute dal modello
# Le classi evidenziate (★) sono quelle rilevanti per questo corso
CLASSI_INTERESSE = {0, 1, 2, 3, 5, 7}  # person, bicycle, car, motorcycle, bus, truck

print(f'Classi COCO: {len(model_yolo.names)} classi totali\n')
print(f"{'ID':>3}  {'Nome':<22}  Note")
print('─' * 40)
for idx in sorted(model_yolo.names.keys()):
    nome = model_yolo.names[idx]
    tag = '  ★ (usata nel corso)' if idx in CLASSI_INTERESSE else ''
    print(f'{idx:>3}  {nome:<22}{tag}')

**Detection su frame singolo — pedoni.** Applichiamo `YOLOv8n` a un frame di MOT20, filtrando solo la classe `person` (ID 0). Il risultato viene visualizzato con bounding box e confidenza tramite `disegna_detection()`.

In [ ]:
# ── 2.1 Detection su frame singolo: pedoni ───────────────────────────────────
frame_pedoni = estrai_frame(VIDEO_PEDONI, n=1)[0]

risultati = model_yolo(frame_pedoni, verbose=False)
frame_annotato = disegna_detection(frame_pedoni, risultati,
                                    soglia_conf=0.3, classi_filtro=[0],
                                    nomi_classi=model_yolo.names)  # classe 0 = person

n_persone = sum(1 for b in risultati[0].boxes if int(b.cls[0]) == 0 and float(b.conf[0]) > 0.3)
print(f'Persone rilevate: {n_persone}')
mostra_confronto(frame_pedoni, frame_annotato, 
                 'Originale', f'Detection ({n_persone} persone)')

**Detection su frame singolo — veicoli.** Stesso approccio su UA-DETRAC, con filtro sulle classi `car`, `bus`, `truck` (ID 2, 5, 7).

In [ ]:
# ── 2.2 Detection su frame singolo: veicoli ──────────────────────────────────
frame_veicoli = estrai_frame(VIDEO_VEICOLI, n=1)[0]

risultati_v = model_yolo(frame_veicoli, verbose=False)
frame_annotato_v = disegna_detection(frame_veicoli, risultati_v,
                                      soglia_conf=0.3, classi_filtro=[2, 5, 7],
                                      nomi_classi=model_yolo.names)  # car, bus, truck

n_veicoli = sum(1 for b in risultati_v[0].boxes
                if int(b.cls[0]) in [2,5,7] and float(b.conf[0]) > 0.3)
print(f'Veicoli rilevati: {n_veicoli}')
mostra_confronto(frame_veicoli, frame_annotato_v,
                 'Originale', f'Detection ({n_veicoli} veicoli)')

### Confronto modelli YOLO: nano · small · medium

I modelli della famiglia YOLOv8 offrono un chiaro **trade-off velocità ↔ accuracy**.
Confrontiamo le 3 versioni più leggere sullo stesso frame:

| Modello | Dimensione | Uso tipico |
|---------|-----------|------------|
| `yolov8n` — **nano** | ~6 MB | real-time su edge / CPU |
| `yolov8s` — **small** | ~22 MB | bilanciamento velocità/qualità |
| `yolov8m` — **medium** | ~52 MB | alta accuracy, GPU consigliata |

> I pesi vengono scaricati automaticamente da Ultralytics al primo utilizzo.

In [ ]:
# ── Confronto YOLOv8: nano · small · medium ───────────────────────────────────
import time

MODELLI_CMP = {
    'yolov8n (nano, ~6MB)':    model_yolo,            # già caricato
    'yolov8s (small, ~22MB)':  YOLO('yolov8s.pt'),
    'yolov8m (medium, ~52MB)': YOLO('yolov8m.pt'),
}
frame_cmp  = estrai_frame(VIDEO_PEDONI, n=1)[0]  # stesso frame per tutti
CLASSI_CMP = [0]   # solo persone
SOGLIA_CMP = 0.3
N_RUN      = 5

risultati_cmp = {}
frames_cmp    = []
titoli_cmp    = []

for nome_mod, modello in MODELLI_CMP.items():
    # Warm-up
    for _ in range(2):
        modello(frame_cmp, verbose=False, classes=CLASSI_CMP)
    # Misura tempo
    tempi = []
    for _ in range(N_RUN):
        t0  = time.perf_counter()
        ris = modello(frame_cmp, verbose=False, classes=CLASSI_CMP)
        tempi.append((time.perf_counter() - t0) * 1000)
    # Statistiche sull'ultimo risultato
    dets      = [b for b in ris[0].boxes
                 if int(b.cls[0]) in CLASSI_CMP and float(b.conf[0]) >= SOGLIA_CMP]
    n_det     = len(dets)
    conf_med  = float(np.mean([float(b.conf[0]) for b in dets])) if dets else 0.0
    tempo_ms  = float(np.mean(tempi))
    risultati_cmp[nome_mod] = {'n_det': n_det, 'conf_media': conf_med, 'tempo_ms': tempo_ms}
    # Frame annotato
    ann = disegna_detection(frame_cmp, ris,
                                   soglia_conf=SOGLIA_CMP,
                                   classi_filtro=CLASSI_CMP,
                                   nomi_classi=modello.names)
    frames_cmp.append(ann)
    tag = nome_mod.split('(')[0].strip()
    titoli_cmp.append(f'{tag} — {n_det} det. · conf {conf_med:.2f} · {tempo_ms:.0f} ms')

mostra_griglia(frames_cmp, titoli_cmp, colonne=2)

In [ ]:
# Tabella plotly
import plotly.graph_objects as go

nomi_m = list(risultati_cmp.keys())
n_dets = [risultati_cmp[n]['n_det']      for n in nomi_m]
confs  = [risultati_cmp[n]['conf_media'] for n in nomi_m]
tempi  = [risultati_cmp[n]['tempo_ms']   for n in nomi_m]

def _col(v, base):
    r = v / base if base > 0 else 1.0
    if r >= 0.97: return 'rgb(198,239,206)'
    if r >= 0.90: return 'rgb(255,235,156)'
    return 'rgb(255,199,206)'

colors_conf = ['white'] + [_col(c, confs[0]) for c in confs[1:]]

fig = go.Figure(data=[go.Table(
    header=dict(
        values=['<b>Modello</b>', '<b>Det. persone</b>', '<b>Conf. media</b>', '<b>Tempo ms/frame</b>'],
        fill_color='#2c3e50', font=dict(color='white', size=13),
        align='center', height=36
    ),
    cells=dict(
        values=[
            nomi_m,
            [str(n) for n in n_dets],
            [f'{c:.3f}' for c in confs],
            [f'{t:.0f}' for t in tempi],
        ],
        fill_color=[
            ['white'] * 3,
            ['white'] * 3,
            colors_conf,
            ['white'] * 3,
        ],
        align='center', height=30, font=dict(size=12)
    )
)])
fig.update_layout(
    title='Confronto YOLOv8 nano · small · medium — stesso frame, soglia conf ≥ 0.3',
    margin=dict(l=10, r=10, t=50, b=10), height=200
)
fig.show()

**Effetto della soglia di confidenza.** Soglia bassa → più detection (inclusi falsi positivi). Soglia alta → solo detection sicure (rischio falsi negativi). Il valore ottimale dipende dal caso d'uso e dal costo relativo dei due tipi di errore.

In [ ]:
# ── 2.3 Effetto della soglia di confidenza ───────────────────────────────────
# Mostriamo come la soglia conf cambia il numero di detection

frame_ref = estrai_frame(VIDEO_PEDONI, n=1)[0]
soglie = [0.1, 0.3, 0.5, 0.7]
frame_confronto = []
titoli_confronto = []

ris = model_yolo(frame_ref, verbose=False)
for s in soglie:
    ann = disegna_detection(frame_ref, ris, soglia_conf=s, classi_filtro=[0], nomi_classi=model_yolo.names)
    n = sum(1 for b in ris[0].boxes if int(b.cls[0]) == 0 and float(b.conf[0]) > s)
    frame_confronto.append(ann)
    titoli_confronto.append(f'conf ≥ {s}  ({n} det.)')

mostra_griglia(frame_confronto, titoli_confronto, colonne=2)

### Domanda
Cosa osservate al variare della soglia di confidenza? Qual è il trade-off tra **falsi positivi** e **falsi negativi**?

*(Discutere con il gruppo prima di procedere)*

In [ ]:
# ── 2.4 Detection su una sequenza di frame (mini-video) ──────────────────────
# Processiamo i primi N frame del video e salviamo il risultato

N_FRAME_DA_PROCESSARE = 150  # ~5 secondi a 30fps
OUTPUT_VIDEO_DET = str(VIDEOS_DIR / 'day1_detection_pedoni.mp4')
OUTPUT_VIDEO_DET_TMP = str(VIDEOS_DIR / 'day1_detection_pedoni_tmp.mp4')
SOGLIA_CONF = 0.35

cap = cv2.VideoCapture(VIDEO_PEDONI)
fps = cap.get(cv2.CAP_PROP_FPS)
w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
writer = cv2.VideoWriter(OUTPUT_VIDEO_DET_TMP, fourcc, fps, (w, h))

n_det_per_frame = []
for i in tqdm(range(N_FRAME_DA_PROCESSARE), desc='Detection'):
    ret, frame = cap.read()
    if not ret:
        break
    ris = model_yolo(frame, verbose=False)
    ann = disegna_detection(frame, ris, soglia_conf=SOGLIA_CONF, classi_filtro=[0])
    n = sum(1 for b in ris[0].boxes if int(b.cls[0]) == 0 and float(b.conf[0]) > SOGLIA_CONF)
    n_det_per_frame.append(n)
    writer.write(ann)

cap.release()
writer.release()

OUTPUT_VIDEO_DET = converti_video_h264(OUTPUT_VIDEO_DET_TMP, OUTPUT_VIDEO_DET)
print(f'Video salvato in: {OUTPUT_VIDEO_DET}')
print(f'Media detection per frame: {np.mean(n_det_per_frame):.1f}')

In [ ]:
# Riproduce il video di detection nel notebook tramite HTML5 embed (base64).
# Mostra il video risultante
mostra_video_colab(OUTPUT_VIDEO_DET)


In [ ]:
# Grafico interattivo: andamento temporale delle detection per frame.
# La linea tratteggiata (media) identifica la densità tipica della scena.
# Plot interattivo del numero di detection nel tempo
import plotly.graph_objects as go

media = np.mean(n_det_per_frame)
fig = go.Figure()
fig.add_trace(go.Scatter(
    y=n_det_per_frame, mode='lines',
    line=dict(color='steelblue', width=1.5),
    name='Detection per frame'
))
fig.add_hline(
    y=media, line_dash='dash', line_color='red',
    annotation_text=f'Media: {media:.1f}', annotation_position='top right'
)
fig.update_layout(
    title='Detection nel tempo — MOT20-01',
    xaxis_title='Frame', yaxis_title='N° persone rilevate',
    height=350, margin=dict(l=50, r=20, t=50, b=40)
)
fig.show()


---
## 3. Multi-Object Tracking con BoxMOT

La **detection** risponde a: *"Dove sono gli oggetti in questo frame?"*  
Il **tracking** risponde a: *"Dove si trova lo stesso oggetto nel frame successivo?"*

**BoxMOT** è una libreria che combina detector (come YOLO) con algoritmi di tracking:
- **ByteTrack**: associa detection anche a bassa confidenza usando la predizione del moto
- **StrongSORT**: usa feature di re-identificazione visiva per tracking robusto
- **BoTrack**, **OCSORT**, e altri

Ogni oggetto tracciato riceve un **ID univoco** che persiste nel tempo.

**Output del tracker**: per ogni frame → `[x1, y1, x2, y2, track_id, conf, cls, ?]`

In [ ]:
from boxmot import ByteTrack
import torch

# Inizializzazione del tracker ByteTrack
# ByteTrack non richiede modelli aggiuntivi (solo motion-based)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device in uso: {device}')

tracker = ByteTrack(
    track_thresh=0.45,   # soglia minima confidenza per tracking
    match_thresh=0.8,    # soglia IoU per associazione
    track_buffer=30,     # frame da attendere prima di eliminare un track perso
)

### 3.1 Selezione interattiva della persona da tracciare

Nella cella seguente vengono mostrate tutte le persone rilevate nel **primo frame** del video.
Usate lo **slider** per scegliere quale persona seguire — senza modificare il codice.

Il video prodotto dalla cella successiva mostrerà:
- Tutti i track in **grigio** (contorno sottile, nessuna etichetta)
- La persona scelta evidenziata con un **bordo colorato** e la label `★ ID:X`
- Un **ritaglio** del soggetto sovrapposto in alto a destra per seguirlo facilmente

In [ ]:
import ipywidgets as widgets
from IPython.display import display as ipy_display

# ── Rileva le persone nel primo frame e mostra lo slider ──────────────────────
_frame_sel = estrai_frame(VIDEO_PEDONI, n=1)[0]
_ris_sel   = model_yolo(_frame_sel, verbose=False, classes=[0])
_dets_sel  = [b for b in _ris_sel[0].boxes
              if int(b.cls[0]) == 0 and float(b.conf[0]) >= 0.3]
N_PERSONE_SEL = len(_dets_sel)
print(f'Persone rilevate nel primo frame: {N_PERSONE_SEL}')

# Frame con detection numerate
_frame_num = _frame_sel.copy()
for idx, b in enumerate(_dets_sel, start=1):
    x1, y1, x2, y2 = map(int, b.xyxy[0])
    cv2.rectangle(_frame_num, (x1, y1), (x2, y2), (0, 255, 0), 2)
    cv2.putText(_frame_num, f'#{idx}', (x1, y1 - 6),
                cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 255, 0), 2)
mostra_frame(_frame_num, f'Persone rilevate ({N_PERSONE_SEL}) — scegli con lo slider')

# Slider interattivo
slider_persona = widgets.IntSlider(
    value=1, min=1, max=max(N_PERSONE_SEL, 1), step=1,
    description='Persona n°:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

def _on_change(change):
    print(f'  ▶  Selezionata persona #{change["new"]}  '
          f'— esegui la cella successiva per generare il video')
slider_persona.observe(_on_change, names='value')
ipy_display(slider_persona)

**Tracking con ByteTrack.** Integriamo le detection YOLO con ByteTrack per assegnare un ID persistente a ogni persona. Il soggetto scelto con lo slider viene evidenziato con ★ e mostrato in un inset in alto a destra del video output.

In [ ]:
# ── 3.2 Tracking con persona selezionata ─────────────────────────────────────
# Regola lo slider nella cella precedente, poi esegui questa per generare il video.
N_FRAME_TRACKING       = 150
OUTPUT_VIDEO_TRACK     = str(VIDEOS_DIR / 'day1_tracking_pedoni.mp4')
OUTPUT_VIDEO_TRACK_TMP = str(VIDEOS_DIR / 'day1_tracking_pedoni_tmp.mp4')
SOGLIA_YOLO            = 0.3

# ── Bounding box della persona scelta nel primo frame ─────────────────────────
persona_scelta = slider_persona.value   # 1-based
_frame_init = estrai_frame(VIDEO_PEDONI, n=1)[0]
_ris_init   = model_yolo(_frame_init, verbose=False, classes=[0])
_dets_init  = [b for b in _ris_init[0].boxes
               if int(b.cls[0]) == 0 and float(b.conf[0]) >= SOGLIA_YOLO]

if persona_scelta <= len(_dets_init):
    _b = _dets_init[persona_scelta - 1]
else:
    print(f'⚠️  Persona #{persona_scelta} non trovata, uso persona #1')
    _b = _dets_init[0]

# Box di riferimento per trovare il track ID al primo aggiornamento
_target_box = np.array(list(map(float, _b.xyxy[0])))  # [x1, y1, x2, y2]

def _iou(a, b):
    """IoU tra due box [x1,y1,x2,y2]."""
    ix1 = max(a[0], b[0]);  iy1 = max(a[1], b[1])
    ix2 = min(a[2], b[2]);  iy2 = min(a[3], b[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    ua = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / ua if ua > 0 else 0.0

# ── Tracking dell'intera sequenza ─────────────────────────────────────────────
# ID_TARGET viene risolto al PRIMO frame del loop principale (stesso tracker,
# stessa istanza) confrontando via IoU con _target_box.
tracker = ByteTrack(track_thresh=0.45, match_thresh=0.8, track_buffer=30)

cap = cv2.VideoCapture(VIDEO_PEDONI)
fps = cap.get(cv2.CAP_PROP_FPS)
w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

writer = cv2.VideoWriter(OUTPUT_VIDEO_TRACK_TMP,
                         cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))
INSET_H, INSET_W    = h // 4, w // 4
id_attivi_per_frame = []
target_trovato_n    = 0
ID_TARGET           = None  # risolto al primo frame

for i in tqdm(range(N_FRAME_TRACKING), desc='Tracking'):
    ret, frame = cap.read()
    if not ret:
        break

    # Detection
    ris  = model_yolo(frame, verbose=False, classes=[0])
    dets = []
    for box in ris[0].boxes:
        conf = float(box.conf[0])
        if conf >= SOGLIA_YOLO:
            x1, y1, x2, y2 = map(float, box.xyxy[0])
            dets.append([x1, y1, x2, y2, conf, float(box.cls[0])])
    dets_np = np.array(dets) if dets else np.empty((0, 6))

    # Aggiornamento tracker
    tracks = tracker.update(dets_np, frame)
    id_attivi_per_frame.append(len(tracks) if tracks is not None else 0)

    # Primo frame: trova ID_TARGET tramite IoU con il box scelto
    if ID_TARGET is None and tracks is not None and len(tracks) > 0:
        best_iou, best_id = 0.0, int(tracks[0][4])
        for trk in tracks:
            v = _iou(_target_box, trk[:4])
            if v > best_iou:
                best_iou, best_id = v, int(trk[4])
        ID_TARGET = best_id
        print(f'Persona #{persona_scelta} → Track ID: {ID_TARGET}  (IoU = {best_iou:.2f})')

    # Disegno: target evidenziato, altri in grigio
    frame_annotato = disegna_tracks(frame, tracks, id_target=ID_TARGET)

    # Inset crop del target in alto a destra (se visibile)
    if tracks is not None and ID_TARGET is not None:
        for trk in tracks:
            if int(trk[4]) == ID_TARGET:
                target_trovato_n += 1
                tx1 = max(int(trk[0]), 0);  ty1 = max(int(trk[1]), 0)
                tx2 = min(int(trk[2]), w);  ty2 = min(int(trk[3]), h)
                if tx2 > tx1 and ty2 > ty1:
                    crop   = frame[ty1:ty2, tx1:tx2]
                    crop_r = cv2.resize(crop, (INSET_W, INSET_H))
                    cv2.rectangle(crop_r, (0, 0), (INSET_W-1, INSET_H-1), (0, 220, 0), 3)
                    ix1 = w - INSET_W - 8
                    frame_annotato[8:8 + INSET_H, ix1:ix1 + INSET_W] = crop_r
                break

    cv2.putText(frame_annotato,
                f'Frame: {i+1} | Target ★ID:{ID_TARGET} | Attivi: {id_attivi_per_frame[-1]}',
                (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    writer.write(frame_annotato)

cap.release()
writer.release()

OUTPUT_VIDEO_TRACK = converti_video_h264(OUTPUT_VIDEO_TRACK_TMP, OUTPUT_VIDEO_TRACK)
print(f'Video salvato: {OUTPUT_VIDEO_TRACK}')
print(f'Target (★ID:{ID_TARGET}) visibile in {target_trovato_n}/{N_FRAME_TRACKING} frame')
print(f'Media track attivi per frame: {np.mean(id_attivi_per_frame):.1f}')

In [ ]:
# Visualizza il video di tracking: ID colorati per soggetto, inset del target ★ in alto a destra.
mostra_video_colab(OUTPUT_VIDEO_TRACK)


In [ ]:
# ── 3.3 Tracking su veicoli (UA-DETRAC) ─────────────────────────────────────
N_FRAME_VEI = 120
OUTPUT_VIDEO_TRACK_VEI = str(VIDEOS_DIR / 'day1_tracking_veicoli.mp4')
OUTPUT_VIDEO_TRACK_VEI_TMP = str(VIDEOS_DIR / 'day1_tracking_veicoli_tmp.mp4')

tracker_vei = ByteTrack()
cap = cv2.VideoCapture(VIDEO_VEICOLI)
fps_v = cap.get(cv2.CAP_PROP_FPS)
wv = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
hv = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
writer = cv2.VideoWriter(OUTPUT_VIDEO_TRACK_VEI_TMP,
                         cv2.VideoWriter_fourcc(*'mp4v'), fps_v, (wv, hv))

for i in tqdm(range(N_FRAME_VEI), desc='Tracking veicoli'):
    ret, frame = cap.read()
    if not ret:
        break
    ris = model_yolo(frame, verbose=False, classes=[2, 5, 7])  # car, bus, truck
    dets = []
    for box in ris[0].boxes:
        conf = float(box.conf[0])
        if conf >= 0.3:
            x1, y1, x2, y2 = map(float, box.xyxy[0])
            dets.append([x1, y1, x2, y2, conf, int(box.cls[0])])
    dets_np = np.array(dets) if dets else np.empty((0, 6))
    tracks = tracker_vei.update(dets_np, frame)
    frame_ann = disegna_tracks(frame, tracks)
    writer.write(frame_ann)

cap.release()
writer.release()

OUTPUT_VIDEO_TRACK_VEI = converti_video_h264(OUTPUT_VIDEO_TRACK_VEI_TMP, OUTPUT_VIDEO_TRACK_VEI)
mostra_video_colab(OUTPUT_VIDEO_TRACK_VEI)

### Osservazioni sul tracking
- Notate come gli **ID rimangano stabili** anche quando un oggetto è parzialmente occluso?
- Cosa succede quando due persone si incrociano? (fenomeno di **ID switch**)
- In quali scene il tracker va in difficoltà?

---
## 4. Degradazione controllata con OpenCV

In scenari reali, le telecamere di sorveglianza spesso producono immagini di qualità ridotta per:
- Bassa risoluzione (telecamere economiche, compressione)
- Sfocatura (messa a fuoco errata, movimento rapido)
- Rumore (bassa illuminazione, sensori datati)

Simuliamo queste condizioni in modo **controllato e riproducibile** usando OpenCV.

**Obiettivo:** capire *quantitativamente* come ogni tipo di degradazione impatta la detection.

In [ ]:
# Frame di riferimento per gli esperimenti di degradazione.
# Rappresenta la qualità 'originale' — stesso frame usato in tutti i confronti successivi.
# Prendiamo un frame di riferimento per gli esperimenti di degradazione
frame_ref = estrai_frame(VIDEO_PEDONI, n=1)[0]
print(f'Frame di riferimento: {frame_ref.shape[1]}x{frame_ref.shape[0]} px')


In [ ]:
# ── 4.1 Riduzione di risoluzione ─────────────────────────────────────────────
scale_factors = [1.0, 0.75, 0.5, 0.25]
frames_scala = [riduci_risoluzione(frame_ref, s) for s in scale_factors]
titoli_scala = [f'Scala {s:.0%}' for s in scale_factors]

mostra_griglia(frames_scala, titoli_scala, colonne=2)

In [ ]:
# ── 4.2 Blur gaussiano ───────────────────────────────────────────────────────
kernel_sizes = [0, 5, 15, 31]
frames_blur = [frame_ref.copy() if k == 0 else applica_blur(frame_ref, k)
               for k in kernel_sizes]
titoli_blur = [f'Kernel {k}x{k}' if k > 0 else 'Originale' for k in kernel_sizes]

mostra_griglia(frames_blur, titoli_blur, colonne=2)

In [ ]:
# ── 4.3 Rumore gaussiano ─────────────────────────────────────────────────────
livelli_rumore = [0, 15, 30, 60]
frames_rumore = [frame_ref.copy() if n == 0 else aggiungi_rumore(frame_ref, n)
                 for n in livelli_rumore]
titoli_rumore = [f'σ={n}' if n > 0 else 'Originale' for n in livelli_rumore]

mostra_griglia(frames_rumore, titoli_rumore, colonne=2)

In [ ]:
# ── 4.4 Degradazione combinata ───────────────────────────────────────────────
# Simuliamo una telecamera di sorveglianza reale a bassa qualità
frame_degradato_lieve  = degrada_immagine(frame_ref, scala=0.75, blur=5,  rumore=15.0)
frame_degradato_medio  = degrada_immagine(frame_ref, scala=0.5,  blur=11, rumore=25.0)
frame_degradato_forte  = degrada_immagine(frame_ref, scala=0.25, blur=21, rumore=40.0)

mostra_griglia(
    [frame_ref, frame_degradato_lieve, frame_degradato_medio, frame_degradato_forte],
    ['Originale', 'Degrado lieve', 'Degrado medio', 'Degrado forte'],
    colonne=2
)

---
## 5. Impatto della degradazione su Detection e Tracking

Ora applichiamo YOLO + ByteTrack alle stesse sequenze con degradazione crescente e **confrontiamo i risultati**.

In [ ]:
# Helper: esegue YOLO su n_frame con degradazione opzionale e calcola statistiche aggregate.
# degrado_fn: lambda opzionale applicata al frame prima della detection (es. blur, scala, rumore).
# Restituisce: media detection/frame, deviazione standard, confidenza media.
def run_detection_su_sequenza(video_path: str, n_frame: int,
                               degrado_fn=None, soglia: float = 0.35,
                               classi: list = [0]) -> dict:
    """
    Esegue YOLO su n_frame di un video, con eventuale degradazione.
    Restituisce statistiche aggregata.
    """
    cap = cv2.VideoCapture(video_path)
    n_det_lista = []
    conf_lista = []

    for _ in range(n_frame):
        ret, frame = cap.read()
        if not ret:
            break
        if degrado_fn is not None:
            frame = degrado_fn(frame)
        ris = model_yolo(frame, verbose=False, classes=classi)
        dets = [b for b in ris[0].boxes
                if int(b.cls[0]) in classi and float(b.conf[0]) >= soglia]
        n_det_lista.append(len(dets))
        conf_lista.extend([float(b.conf[0]) for b in dets])

    cap.release()
    return {
        'media_det': np.mean(n_det_lista) if n_det_lista else 0,
        'std_det': np.std(n_det_lista) if n_det_lista else 0,
        'media_conf': np.mean(conf_lista) if conf_lista else 0,
        'n_det_lista': n_det_lista,
    }


**Test sistematico degli scenari.** Eseguiamo YOLO su 8 scenari di degradazione (100 frame ciascuno) per *quantificare* l'impatto di ogni tipo di degrado su numero e qualità delle detection. I risultati sono poi confrontati in tabella e grafici.

In [ ]:
# Definizione degli scenari di test
N_FRAME_TEST = 100

scenari = {
    'Originale':       None,
    'Scala 75%':       lambda f: riduci_risoluzione(f, 0.75),
    'Scala 50%':       lambda f: riduci_risoluzione(f, 0.50),
    'Scala 25%':       lambda f: riduci_risoluzione(f, 0.25),
    'Blur lieve':      lambda f: applica_blur(f, 7),
    'Blur forte':      lambda f: applica_blur(f, 21),
    'Rumore σ=25':     lambda f: aggiungi_rumore(f, 25),
    'Degrado medio':   lambda f: degrada_immagine(f, 0.5, 11, 25),
}

risultati_scenari = {}
for nome, fn in scenari.items():
    print(f'  Elaborazione: {nome}...')
    risultati_scenari[nome] = run_detection_su_sequenza(
        VIDEO_PEDONI, N_FRAME_TEST, degrado_fn=fn
    )

print('\n✅ Completato')


In [ ]:
# ── Tabella interattiva dei risultati ────────────────────────────────────────
import plotly.graph_objects as go
import pandas as pd

nomi        = list(risultati_scenari.keys())
medie       = [risultati_scenari[n]['media_det'] for n in nomi]
stds        = [risultati_scenari[n]['std_det']   for n in nomi]
confs       = [risultati_scenari[n]['media_conf'] for n in nomi]
baseline_det  = medie[0]
baseline_conf = confs[0]

# Colore cella in base al calo rispetto alla baseline
def _cella_color(val, baseline, invert=False):
    ratio = val / baseline if baseline > 0 else 1.0
    if invert:
        ratio = 1 - ratio + 1
    if ratio >= 0.9:
        return 'rgb(198,239,206)'   # verde
    elif ratio >= 0.7:
        return 'rgb(255,235,156)'   # giallo
    else:
        return 'rgb(255,199,206)'   # rosso

colors_det  = ['white'] + [_cella_color(m, baseline_det)  for m in medie[1:]]
colors_conf = ['white'] + [_cella_color(c, baseline_conf) for c in confs[1:]]

fig = go.Figure(data=[go.Table(
    header=dict(
        values=['<b>Scenario</b>', '<b>Media det/frame</b>', '<b>Dev. std</b>', '<b>Conf. media</b>'],
        fill_color='#2c3e50', font=dict(color='white', size=13),
        align='center', height=36
    ),
    cells=dict(
        values=[
            nomi,
            [f'{m:.1f}' for m in medie],
            [f'{s:.1f}' for s in stds],
            [f'{c:.3f}' for c in confs],
        ],
        fill_color=[
            ['white'] * len(nomi),
            colors_det,
            ['white'] * len(nomi),
            colors_conf,
        ],
        align='center', height=30,
        font=dict(size=12)
    )
)])
fig.update_layout(
    title='Impatto della degradazione su YOLO — riepilogo (verde=ok, giallo=lieve, rosso=severo)',
    margin=dict(l=10, r=10, t=50, b=10), height=360
)
fig.show()

In [ ]:
# Grafici interattivi comparativi
from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=('Numero di detection per scenario',
                                    'Confidenza media delle detection'))

# --- Grafico sinistra: media detection ---
fig.add_trace(
    go.Bar(
        y=nomi, x=medie,
        error_x=dict(type='data', array=stds),
        orientation='h',
        marker_color='steelblue', opacity=0.85,
        name='Media det/frame'
    ), row=1, col=1
)
fig.add_vline(x=baseline_det, line_dash='dash', line_color='red',
              annotation_text='Baseline', annotation_position='top right',
              row=1, col=1)

# --- Grafico destra: confidenza media ---
fig.add_trace(
    go.Bar(
        y=nomi, x=confs,
        orientation='h',
        marker_color='darkorange', opacity=0.85,
        name='Conf. media'
    ), row=1, col=2
)
fig.add_vline(x=baseline_conf, line_dash='dash', line_color='red',
              annotation_text='Baseline', annotation_position='top right',
              row=1, col=2)

fig.update_layout(
    height=420, showlegend=False,
    margin=dict(l=10, r=10, t=60, b=10)
)
fig.update_xaxes(title_text='Media det/frame', row=1, col=1)
fig.update_xaxes(title_text='Confidenza media', row=1, col=2)
fig.show()

**Confronto visivo finale.** Stessa scena, 4 condizioni: originale · scala 50% · blur forte · degrado combinato. Sintesi visiva dell'impatto della degradazione su YOLO.

In [ ]:
# ── Confronto visivo: detection originale vs. degradato ──────────────────────
frame_test = estrai_frame(VIDEO_PEDONI, n=1)[0]

confronti = [
    (frame_test, 'Originale'),
    (riduci_risoluzione(frame_test, 0.5), 'Scala 50%'),
    (applica_blur(frame_test, 21), 'Blur forte'),
    (degrada_immagine(frame_test, 0.5, 11, 25), 'Degrado medio'),
]

frames_vis = []
titoli_vis = []
for img, nome in confronti:
    ris = model_yolo(img, verbose=False, classes=[0])
    ann = disegna_detection(img, ris, soglia_conf=0.3, classi_filtro=[0], nomi_classi=model_yolo.names)
    n = sum(1 for b in ris[0].boxes if int(b.cls[0]) == 0 and float(b.conf[0]) > 0.3)
    frames_vis.append(ann)
    titoli_vis.append(f'{nome} ({n} det.)')

mostra_griglia(frames_vis, titoli_vis, colonne=2)

---
## 6. Riepilogo e discussione

### Cosa abbiamo fatto oggi:
1. **Esplorato** tre tipologie di dataset di sorveglianza (pedoni, veicoli, folla)
2. **Applicato** YOLOv8 per la detection di persone e veicoli
3. **Implementato** il Multi-Object Tracking con ByteTrack
4. **Simulato** degradazione controllata con OpenCV
5. **Quantificato** l'impatto della degradazione su detection e tracking

### Punti chiave:
- La **riduzione di risoluzione** è spesso più impattante del blur
- Il **tracker** può compensare parzialmente le mancate detection (usando la predizione del moto)
- La **soglia di confidenza** è un parametro critico: va calibrata in base al caso d'uso

### Domande di riflessione:
- In quale scenario il tracking è più utile rispetto alla sola detection?
- Come gestireste un sistema in produzione dove la qualità del video varia?
- Quale tipo di degradazione è più difficile da gestire per YOLO?

In [ ]:
# Esercizio libero: modifica VIDEO_SCELTO, SOGLIA_SCELTA e DEGRADO_SCELTO per sperimentare.
# Prova diversi video e combinazioni di soglia + degradazione per osservare i risultati.
# ── ESERCIZIO FINALE: prova con un altro video ───────────────────────────────
# Scegliete un video dalla lista, cambiate i parametri e osservate i risultati

VIDEO_SCELTO = VIDEO_FOLLA  # ← modificate qui
SOGLIA_SCELTA = 0.35        # ← provate valori diversi
DEGRADO_SCELTO = None       # ← provate: lambda f: degrada_immagine(f, 0.5, 11, 20)

frame_es = estrai_frame(VIDEO_SCELTO, n=1)[0]
if DEGRADO_SCELTO:
    frame_es_deg = DEGRADO_SCELTO(frame_es)
else:
    frame_es_deg = frame_es

ris_es = model_yolo(frame_es_deg, verbose=False)
frame_es_ann = disegna_detection(frame_es_deg, ris_es, soglia_conf=SOGLIA_SCELTA, nomi_classi=model_yolo.names)
n_es = sum(1 for b in ris_es[0].boxes if float(b.conf[0]) > SOGLIA_SCELTA)

print(f'Detection trovate: {n_es}')
mostra_confronto(frame_es, frame_es_ann, 'Originale', f'Detection (soglia={SOGLIA_SCELTA})')
